# 05. Dla ciekawych: formatowanie, strefy czasowe, sekundy przestępne

## Cele

Ten notebook to materiał dodatkowy, bez ćwiczenia. Po jego przeczytaniu:

* wiesz, jak wypisać datę i czas przez `std::format` i `operator<<`,
* rozumiesz różnicę między czasem UTC (`sys_time`) a czasem lokalnym (`local_time`, `zoned_time`),
* wiesz, co dzieje się z „lokalną godziną” przy zmianie czasu letniego — i dlaczego może jej nie być albo być dwie,
* znasz sekundę przestępną `23:59:60`, problem roku 2038 i kilka głośnych awarii związanych z czasem.

> **Wymaga nowszego kompilatora.** Przykłady z tego notebooka nie skompilują się na GCC 11 ani Clang 14.
> Formatowanie i strefy czasowe są w libstdc++ od **GCC 13**, `parse` od **GCC 14**
> ([status libstdc++](https://gcc.gnu.org/onlinedocs/libstdc++/manual/status.html#status.iso.2020));
> w MSVC od Visual Studio 2019 16.10, przy czym `<format>` w trybie `/std:c++20` od VS 2022 17.2
> ([Microsoft: conformance](https://learn.microsoft.com/en-us/cpp/overview/visual-cpp-language-conformance?view=msvc-170#c-standard-library-features)).
> W libc++ (Clang) strefy czasowe są wciąż eksperymentalne, a `parse` jeszcze go nie ma
> ([cppreference: compiler support](https://en.cppreference.com/cpp/compiler_support/20#P0355R7)).
> Jeśli masz starszy kompilator, uruchom przykłady w [Compiler Explorer](https://godbolt.org/) z najnowszym GCC.

## Wypisywanie: `operator<<` i `std::format`

Od C++20 typy `<chrono>` można wypisywać wprost. Odcinek czasu wypisuje się jako liczba z sufiksem jednostki
([cppreference: `operator<<(duration)`](https://en.cppreference.com/cpp/chrono/duration/operator_ltlt)),
punkt w czasie — jako data i godzina
([cppreference: `operator<<(sys_time)`](https://en.cppreference.com/cpp/chrono/system_clock/operator_ltlt)).
`std::format` przyjmuje specyfikatory podobne do `strftime`: `%F` to `%Y-%m-%d`, `%T` to `%H:%M:%S`,
`%Z` to nazwa strefy, `%z` to przesunięcie względem UTC
([cppreference: specyfikatory](https://en.cppreference.com/cpp/chrono/system_clock/formatter#Format_specification)).

```cpp
#include <chrono>
#include <format>
#include <iostream>

int main() {
    using namespace std::chrono;
    std::cout << 90min << ", " << 1500ms << ", " << 2026y / November / 9d << '\n';

    sys_seconds tp = sys_days{2026y / November / 9d} + 10h + 15min + 30s;
    std::cout << tp << '\n';
    std::cout << std::format("{:%F %T %Z}\n", tp);
    std::cout << std::format("{:%Y-%m-%d %H:%M}\n", tp);
    std::cout << std::format("dzień roku {:%j}, tydzień ISO {:%V}\n", tp, tp);
}
```

Integracja `<chrono>` z `std::format` przyszła w ostatniej chwili przed C++20
([P1361R2](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2019/p1361r2.pdf)) — stąd późne wsparcie w kompilatorach.

## Strefy czasowe: UTC kontra czas lokalny

`system_clock` (i `sys_time`) to zawsze UTC. Czas „na zegarku w Warszawie” to co innego:

* `local_time` — godzina lokalna „w jakiejś, jeszcze nieokreślonej strefie”
  ([cppreference: `local_t`](https://en.cppreference.com/cpp/chrono/local_t)),
* `time_zone` — reguły konkretnej strefy z bazy IANA (`locate_zone("Europe/Warsaw")`)
  ([cppreference: `tzdb`](https://en.cppreference.com/cpp/chrono/tzdb)),
* `zoned_time` — para: strefa + punkt w czasie
  ([cppreference: `zoned_time`](https://en.cppreference.com/cpp/chrono/zoned_time)).

```cpp
#include <chrono>
#include <format>
#include <iostream>

int main() {
    using namespace std::chrono;
    const time_zone* warsaw = locate_zone("Europe/Warsaw");
    sys_seconds utc = sys_days{2026y / November / 9d} + 10h + 15min + 30s;

    zoned_time in_warsaw{warsaw, utc};
    std::cout << std::format("UTC:      {:%F %T}\n", utc);
    std::cout << std::format("Warszawa: {:%F %T %Z %z}\n", in_warsaw);
    std::cout << "baza stref: " << get_tzdb().version << '\n';
}
```

W przykładach używamy `locate_zone`, a nie `current_zone()`: strefa bieżącego komputera bywa różna
(na serwerach często UTC), a niektóre implementacje nie potrafią jej ustalić.

## Zmiana czasu: godzina, której nie ma, i godzina, która jest dwa razy

W Polsce w 2026 roku czas letni zaczyna się **29 marca** (zegary z 2:00 na 3:00) i kończy **25 października**
(z 3:00 na 2:00) — obie zmiany o 1:00 UTC. Tak stanowi
[rozporządzenie Prezesa Rady Ministrów (Dz.U. 2022 poz. 539)](https://api.sejm.gov.pl/eli/acts/DU/2022/539/text.pdf) [PL];
ciekawostka z jego tekstu: powtórzona jesienią godzina jest tam oznaczona jako „2a”.
Te same reguły zapisane są w bazie stref IANA
([plik `europe`](https://raw.githubusercontent.com/eggert/tz/main/europe), [Wikipedia: Czas letni](https://pl.wikipedia.org/wiki/Czas_letni#Czas_letni_w_Polsce) [PL]).

Wynika z tego, że:

* **29 marca 2026, 2:30** w Warszawie **nie istnieje** — zamiana na UTC rzuca `nonexistent_local_time`,
* **25 października 2026, 2:30** zdarza się **dwa razy** — zamiana rzuca `ambiguous_local_time`

([cppreference: `to_sys`](https://en.cppreference.com/cpp/chrono/time_zone/to_sys#Notes)).
Żeby wybrać jedną z możliwości, podaje się `choose::earliest` albo `choose::latest`
([cppreference: `choose`](https://en.cppreference.com/cpp/chrono/choose)):

```cpp
#include <chrono>
#include <format>
#include <iostream>

int main() {
    using namespace std::chrono;
    const time_zone* warsaw = locate_zone("Europe/Warsaw");

    local_seconds spring = local_days{2026y / March / 29d} + 2h + 30min;
    try {
        zoned_time z{warsaw, spring};
    } catch (const nonexistent_local_time&) {
        std::cout << "2026-03-29 02:30 w Warszawie nie istnieje\n";
    }

    local_seconds autumn = local_days{2026y / October / 25d} + 2h + 30min;
    zoned_time first{warsaw, autumn, choose::earliest};
    zoned_time second{warsaw, autumn, choose::latest};
    std::cout << std::format("pierwsze 2:30: {:%T %Z} = {:%T} UTC\n", first, first.get_sys_time());
    std::cout << std::format("drugie 2:30:   {:%T %Z} = {:%T} UTC\n", second, second.get_sys_time());
    std::cout << "między nimi: " << (second.get_sys_time() - first.get_sys_time()) << '\n';
}
```

Doba 29 marca 2026 trwa w Warszawie 23 godziny, a 25 października — 25. Dlatego „dodaj dzień” na czasie lokalnym
i „dodaj 24 godziny” na UTC to dwie różne operacje.

## Sekundy przestępne i `utc_clock`

Ziemia nie obraca się idealnie równo, więc od 1972 roku do UTC czasem dodawano **sekundę przestępną**:
po `23:59:59` następowało `23:59:60`. Łącznie dodano ich 27, ostatnią 31 grudnia 2016
([IERS Bulletin C 52](https://hpiers.obspm.fr/iers/bul/bulc/bulletinc.52),
[Wikipedia](https://pl.wikipedia.org/wiki/Sekunda_przest%C4%99pna#W_Polsce) [PL]).

`system_clock` sekund przestępnych **nie liczy** (czas uniksowy). Liczy je `utc_clock`
([cppreference: `utc_clock`](https://en.cppreference.com/cpp/chrono/utc_clock)),
a formatowanie potrafi wypisać sekundę „60”
([cppreference](https://en.cppreference.com/cpp/chrono/utc_clock/formatter)):

```cpp
#include <chrono>
#include <format>
#include <iostream>

int main() {
    using namespace std::chrono;
    utc_seconds u = utc_clock::from_sys(sys_days{2016y / December / 31d} + 23h + 59min + 59s);
    for (int i = 0; i < 3; ++i, u += 1s) {
        auto info = get_leap_second_info(u);
        std::cout << std::format("{:%F %T}", u) << "  sekunda przestępna: " << info.is_leap_second << '\n';
    }
    std::cout << "sekund przestępnych w bazie: " << get_tzdb().leap_seconds.size() << '\n';
}
```

Są też zegary `tai_clock` i `gps_clock`, a `clock_cast` przelicza punkty między zegarami
([cppreference: `clock_cast`](https://en.cppreference.com/cpp/chrono/clock_cast)).

**Sekundy przestępne odchodzą do historii.** Psuły systemy komputerowe (patrz niżej), więc w 2022 roku
Generalna Konferencja Miar zdecydowała, że najpóźniej w 2035 przestaną być dodawane
([BIPM, rezolucja 4 z 2022](https://www.bipm.org/en/cgpm-2022/resolution-4)). **15 października 2026** — kilka tygodni
przed tym laboratorium — przyspieszyła ten termin: *„continuous UTC will become effective on 20 May 2027”*, a różnica
między UTC a czasem astronomicznym może urosnąć do godziny
([BIPM, rezolucja 5 z 2026](https://www.bipm.org/en/cgpm-2026/resolution-5),
[Wikipedia: Sekunda przestępna](https://pl.wikipedia.org/wiki/Sekunda_przest%C4%99pna#Decyzje_z_lat_2022%E2%80%932026) [PL]).
IERS zapowiedział już, że na koniec 2026 roku sekundy przestępnej nie będzie
([Bulletin C 72](https://hpiers.obspm.fr/iers/bul/bulc/bulletinc.72)). Wszystko wskazuje więc na to, że `23:59:60`
z 2016 roku było ostatnim. Do tego czasu duże firmy radziły sobie „rozmazując” dodatkową sekundę na 24 godziny
([Google: Leap Smear](https://developers.google.com/time/smear#standardsmear)).

## Wczytywanie: `parse` (GCC 14+)

Odwrotność formatowania to `std::chrono::parse`. Błąd wczytywania nie rzuca wyjątku, tylko ustawia `failbit`
strumienia ([cppreference: `parse`](https://en.cppreference.com/cpp/chrono/parse#Format_string)):

```cpp
#include <chrono>
#include <iostream>
#include <sstream>

int main() {
    using namespace std::chrono;
    std::istringstream in{"2026-03-29 14:05:09 +0200"};
    sys_seconds tp;
    in >> parse("%F %T %z", tp);  // %z: przesunięcie strefy, wynik w UTC
    if (in) {
        std::cout << "UTC: " << tp << '\n';
    }

    std::istringstream bad{"2026-02-30"};
    year_month_day d;
    bad >> parse("%F", d);
    std::cout << "2026-02-30 wczytane? " << !bad.fail() << '\n';
}
```

Dobra praktyka przy zapisie i przesyłaniu czasu: przechowuj UTC, a jako tekst używaj formatu ISO 8601 / RFC 3339
z jawnym przesunięciem (`2026-03-29T12:05:09Z`) albo liczby sekund od epoki.

## Problem roku 2038

W wielu starszych systemach `time_t` to 32-bitowa liczba sekund od 1970 roku. Jej maksimum to
**19 stycznia 2038, 03:14:07 UTC**; sekundę później licznik „zawija się” na 13 grudnia 1901
([Wikipedia: Year 2038 problem](https://en.wikipedia.org/wiki/Year_2038_problem),
[Problem roku 2038](https://pl.wikipedia.org/wiki/Problem_roku_2038) [PL]):

```cpp
#include <chrono>
#include <cstdint>
#include <iostream>
#include <limits>

int main() {
    using namespace std::chrono;
    std::int32_t t32 = std::numeric_limits<std::int32_t>::max();
    std::cout << sys_seconds{seconds{t32}} << '\n';
    std::int32_t wrapped = static_cast<std::int32_t>(std::int64_t{t32} + 1);  // co zrobi 32-bitowy licznik
    std::cout << sys_seconds{seconds{wrapped}} << '\n';
}
```

Linux od wersji 5.6 i glibc od 2.34 wspierają 64-bitowy `time_t` także na 32-bitowych platformach
([ogłoszenie glibc 2.34](https://sourceware.org/pipermail/libc-alpha/2021-August/129718.html)),
a Debian 13 przeszedł na niego na prawie wszystkich architekturach
([Debian trixie](https://www.debian.org/releases/trixie/release-notes/whats-new.en.html#time64)).
`std::chrono::seconds` ma co najmniej 35 bitów, więc ten problem go nie dotyczy.

## Głośne awarie

* **Patriot, 1991.** W baterii rakiet Patriot w Dhahranie czas był przeliczany z obcięciem w 24-bitowym rejestrze
  stałoprzecinkowym. Błąd rósł z każdą godziną pracy: po 100 godzinach wewnętrzny zegar „rozjechał się” o jedną trzecią
  sekundy; rakieta nie przechwyciła pocisku, zginęło 28 żołnierzy
  ([Wikipedia](https://en.wikipedia.org/wiki/MIM-104_Patriot#Failure_at_Dhahran),
  [raport GAO](https://www.gao.gov/products/imtec-92-26)). Morał: nie sumuj czasu w niedokładnych liczbach.
* **Zune, 31 grudnia 2008.** Odtwarzacze Zune 30 zawiesiły się o północy. Pętla przeliczająca dni na lata
  nie obsługiwała 366. dnia roku przestępnego i kręciła się w nieskończoność
  ([kod sterownika](https://davehope.co.uk/Blog/zune-30gb-bug-sourcecode/),
  [Wikipedia](https://en.wikipedia.org/wiki/Zune_30#Freescale_Driver_Issue)).
* **Windows Azure, 29 lutego 2012.** Data ważności certyfikatu = „ten sam dzień za rok” = 29 lutego 2013
  ([Microsoft](https://azure.microsoft.com/en-us/blog/summary-of-windows-azure-service-disruption-on-feb-29th-2012/)) —
  ten błąd odtworzyliśmy w notebooku 03.
* **Sekunda przestępna, 30 czerwca 2012.** Błąd w jądrze Linuksa przy obsłudze sekundy przestępnej zablokował serwery
  m.in. systemu odpraw linii lotniczych — pasażerów Qantas odprawiano ręcznie
  ([The Register](https://www.theregister.com/software/2012/07/02/leap-second-bug-cripples-linux-servers-at-airlines-reddit-linkedin/826980),
  [poprawka w LKML](https://lkml.iu.edu/hypermail/linux/kernel/1207.2/00960.html)).
* **Cloudflare, 1 stycznia 2017.** Różnica dwóch odczytów zegara systemowego wyszła ujemna — *„the belief that time cannot go backwards”*
  ([Cloudflare](https://blog.cloudflare.com/how-and-why-the-leap-second-affected-cloudflare-dns/)). Pomiar `steady_clock`
  (notebook 02) byłby odporny na ten błąd.

## Przed C++20: biblioteka *date*

Kalendarz i strefy czasowe z C++20 powstały z biblioteki Howarda Hinnanta *date*
([GitHub](https://github.com/HowardHinnant/date#standardization),
[P0355R7](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2018/p0355r7.html)).
Działa ona z C++11/14/17, a kalendarz i formatowanie mieszczą się w jednym nagłówku `date.h`
([dokumentacja](https://howardhinnant.github.io/date/date.html)); strefy czasowe wymagają dodatkowo skompilowania `tz.cpp`
([instalacja](https://howardhinnant.github.io/date/tz.html#Installation)). To dobre wyjście, gdy kompilator nie ma
jeszcze pełnego C++20 — z jednym zastrzeżeniem: w jednym pliku nie otwieraj jednocześnie `using namespace date;`
i `using namespace std::chrono;`, bo nazwy się powtarzają i wywołania stają się niejednoznaczne.

## Linki i materiały

* [cppreference: `<chrono>`](https://en.cppreference.com/w/cpp/chrono.html) — sekcje o kalendarzu i strefach czasowych
* Howard Hinnant, [„Design Rationale for the chrono Library”, Meeting C++ 2019](https://www.youtube.com/watch?v=adSAN282YIw)
  ([slajdy](https://meetingcpp.com/mcpp/slides/2019/Hinnant.pdf)) — przegląd całego `<chrono>` z C++20
* [IANA Time Zone Database](https://www.iana.org/time-zones)
* [IANA: lista sekund przestępnych](https://data.iana.org/time-zones/tzdb/leap-seconds.list)
* [LWN: Approaching the kernel year-2038 end game](https://lwn.net/Articles/776435/)

---

[Strona główna](../_start.ipynb)